In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import gc
import os
import sys
import gymnasium as gym
import torch

torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import MatchEnv, PoolController, RandomController
from src.rl.env_wrapper import Stage2OpponentController
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import RandomReset
from src.rl.reward_shapers import DenseReward_2
from src.rl.trainer import train_ppo

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

# ── CONFIGURATION ──
STAGE = 2  # 1: vs Random, 2: vs Heuristic, 3: vs Mixed Pool
LEARNER_TEAM = "red"
OPPONENT_TEAM = "blue"

MAX_STEPS = 1800  # 30.0s at 60 Hz
TIME_LIMIT = 30.0

SAVE_DIR = f"models/stage{STAGE}"
POOL_DIR = f"models/stage{STAGE}/pool" if STAGE == 3 else None
os.makedirs(SAVE_DIR, exist_ok=True)


def make_env():
    roster = [
        PlayerSlot(LEARNER_TEAM, PlayerStats(name="Learner", accel=3200.0), controller="RL"),
    ]

    if STAGE == 1:
        opp_ctrl = RandomController()
    elif STAGE == 2:
        opp_ctrl = Stage2OpponentController(TeamHeuristicCoordinator(OPPONENT_TEAM))
    else:
        opp_ctrl = PoolController(pool_dir=POOL_DIR, device="cpu")

    roster.append(PlayerSlot(OPPONENT_TEAM, PlayerStats(name="Opponent", accel=3200.0), controller=opp_ctrl))

    cfg = MatchConfig(mode=ClassicMatchMode(time_limit=TIME_LIMIT, score_limit=99), roster=roster)

    return MatchEnv(
        match_config=cfg,
        reward_shaper=DenseReward_2(team=LEARNER_TEAM),
        reset_strategy=RandomReset(),
        learner_team=LEARNER_TEAM,
        max_steps=MAX_STEPS,
    )


NUM_ENVS = 32
train_envs = gym.vector.SyncVectorEnv([make_env for _ in range(NUM_ENVS)])
model = ActorCritic(obs_dim=80).to(device)

if STAGE > 1:
    prev_ckpt = f"models/stage{STAGE-1}/best_model.pt"
    if os.path.exists(prev_ckpt):
        model.load_state_dict(torch.load(prev_ckpt, map_location=device, weights_only=False))
        print(f"✅ Loaded checkpoint from Stage {STAGE-1}")

⚡ Device: cuda
✅ Loaded checkpoint from Stage 2


In [6]:
train_ppo(
    envs=train_envs,
    model=model,
    device=device,
    max_steps=MAX_STEPS,
    time_limit=TIME_LIMIT,
    baseline_type="heuristic",
    total_timesteps=50_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,
    eval_freq=100_000,
    eval_episodes=50,
    save_dir=SAVE_DIR,
    pool_dir=POOL_DIR,
    lr_initial=3e-5,
    lr_final=5e-6,
    gamma=0.999,
    gae_lambda=0.95,
    ent_coef_initial=0.01,  
    ent_coef_final=0.003, 
)

train_envs.close()

🚀 Training: Target [HEURISTIC] | Batch: 8192 | Eval: 50 eps

📊 [EVALUATION @ Step  106496 | Target: HEURISTIC]
   Scoring Episodes: 33/50 (66.0%)
   Goals [Scored: 61 | Conceded: 23 | Net: +38]
   Avg Speed to 1st Goal: 15.35s | Mean Reward: 62.86
   ⭐⭐ PROMOTED! New Best Score: (38, 33, 61, 62.86) -> Saved: models/stage2/best_model.pt

📊 [EVALUATION @ Step  204800 | Target: HEURISTIC]
   Scoring Episodes: 27/50 (54.0%)
   Goals [Scored: 41 | Conceded: 27 | Net: +14]
   Avg Speed to 1st Goal: 15.46s | Mean Reward: 13.23
   ❌ Retaining best model. (Best Score: (38, 33, 61, 62.86))

📊 [EVALUATION @ Step  303104 | Target: HEURISTIC]
   Scoring Episodes: 25/50 (50.0%)
   Goals [Scored: 37 | Conceded: 21 | Net: +16]
   Avg Speed to 1st Goal: 14.86s | Mean Reward: 11.74
   ❌ Retaining best model. (Best Score: (38, 33, 61, 62.86))

📊 [EVALUATION @ Step  401408 | Target: HEURISTIC]
   Scoring Episodes: 21/50 (42.0%)
   Goals [Scored: 30 | Conceded: 19 | Net: +11]
   Avg Speed to 1st Goal: 14.7

In [33]:
import os
import sys
import torch

sys.path.insert(0, os.path.abspath(".."))
from src.rl.evaluator import evaluate_and_generate_html

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Render 5 evaluation matches of your best checkpoint
replay_file = evaluate_and_generate_html(
    model_or_path="models/stage2/best_model.pt",
    device=device,
    baseline_type="heuristic",  # Visualizes matches against Heuristic bot
    output_dir="render/",
    filename="stage2_diagnostic.html",
    num_episodes=10,
    max_steps=1800,  # 30 seconds per match
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/training/render/stage2_diagnostic.html
